### 0.Sentence

`low lower new widest wild`

### 1. BPE - Byte Pair Encoder
**Explanation:**
We want to learn subword units using the BPE algorithm. Here’s the basic idea:

1. Initialize the vocabulary with all the unique characters in the corpus.
- For example, the characters might be:
- `l, o, w,  , e, r, n, i, d, s, t`
- (Assuming we treat spaces as part of the data or we split on spaces first—details vary in real systems.)
2. Split each word into characters (with a special end-of-word symbol or just a whitespace split for demonstration). For example, the word "lower" is split into:
- `["l", "o", "w", "e", "r"]`
3. Count the frequency of adjacent pairs of symbols (initially characters).
- For example, if `"l" "o"` (adjacent in `"lo"`) appears 5 times in the entire corpus, `"o" "w"` appears 3 times, etc.
4. Find the most frequent pair of symbols and merge them into a new symbol (a subword). Add this new subword to your “vocabulary”.
5. Replace all occurrences of that pair in the corpus with the newly merged symbol.
6. Repeat steps 3–5 until you reach a certain number of merges or a desired vocabulary size.

In essence, BPE merges the most frequent pairs of symbols or subwords first, thereby creating longer and longer subwords that are common in the dataset.

**Concrete Mini Example**

Let's assume we do just 2 merges (for demonstration) on the short text. Start with each word split by characters:
- low -> [l, o, w]
- lower -> [l, o, w, e, r]
- new -> [n, e, w]
- widest -> [w, i, d, e, s, t]
- wild -> [w, i, l, d]
First, count pair frequencies in the entire corpus:
- Pair (l, o) in [l, o, w], [l, o, w, e, r]
- Pair (o, w) in [l, o, w], [l, o, w, e, r]
- Pair (w, e) in [l, o, w, e, r], [w, i, d, e, s, t] (but not adjacent in the second case if we skip i?), etc.
We pick the most frequent pair. (Let’s pretend (l, o) is the most frequent, for illustration.)

Merge (l, o) -> new symbol lo. Now the words transform:
- low -> [lo, w]
- lower -> [lo, w, e, r]
- new -> [n, e, w]
- widest -> [w, i, d, e, s, t]
- wild -> [w, i, l, d]
Recount pairs. Let's say the next most frequent pair is (lo, w) (which appears in [lo, w] and [lo, w, e, r]). So we merge (lo, w) -> low.

Now the words transform:
- low -> [low]
- lower -> [low, e, r]
- new -> [n, e, w]
- widest -> [w, i, d, e, s, t]
- wild -> [w, i, l, d]
After these 2 merges, our new “tokens” (subwords) might include:
- `["l", "o", "w", "lo", "low", "e", "r", "n", "i", "d", "s", "t", ...]`

If we want to tokenize a new word lower using our BPE merges, we iteratively find the largest subwords that appear in our BPE merges:
- Start: ["l", "o", "w", "e", "r"]
- We know ["l", "o"] => lo, so we get ["lo", "w", "e", "r"]
- Then ["lo", "w"] => low, so we get ["low", "e", "r"]
- End result: ["low", "e", "r"]
That’s the basic idea.

In [7]:
import collections


def get_vocab(words):
    """
    Convert a list of tokens (words as lists of chars/subwords)
    into a dictionary of subword 'words' -> frequency.
    """
    vocab = collections.Counter()
    for w in words:
        vocab[" ".join(w)] += 1  # ' '.join(w) to keep adjacency explicit
    return vocab


def get_stats(vocab):
    """
    Count frequency of all adjacent pairs of symbols in the current vocab.
    """
    pairs = collections.Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pairs[pair] += freq
    return pairs


def merge_vocab(pair, vocab):
    """
    Merge the given pair in the vocabulary and return the updated vocab.
    """
    bigram = " ".join(pair)
    new_vocab = {}
    for word, freq in vocab.items():
        # Replace all occurrences of the bigram with the merged symbol
        new_word = word.replace(bigram, "".join(pair))
        new_vocab[new_word] = freq
    return new_vocab


def learn_bpe(corpus, num_merges=2):
    """
    Learn BPE merges from a given corpus (list of words) with a specified number of merges.
    Return the merges and the final vocabulary.
    """
    # 1) Split words into list of characters
    tokenized_words = [list(word) for word in corpus]

    # 2) Get initial vocabulary
    vocab = get_vocab(tokenized_words)

    merges = []
    for i in range(num_merges):
        # 3) Get pair stats
        pairs = get_stats(vocab)
        if not pairs:
            break
        # 4) Get most frequent pair
        best_pair = max(pairs, key=pairs.get)
        merges.append(best_pair)

        # 5) Merge in vocab
        vocab = merge_vocab(best_pair, vocab)

    return merges, vocab


def apply_bpe(word, merges):
    """
    Tokenize a single word using the learned BPE merges (greedy left-to-right).
    """
    symbols = list(word)
    i = 0
    while i < (len(symbols) - 1):
        # Check adjacent pair
        pair = (symbols[i], symbols[i + 1])
        if pair in merges:
            # Merge them
            symbols[i : i + 2] = ["".join(pair)]
            # Don't increment i, because we might merge newly formed tokens further
        else:
            i += 1
    return symbols


# --- Example usage ---
corpus = ["low", "lower", "new", "wildest", "wild"]
merges, final_vocab = learn_bpe(corpus, num_merges=5)
print("Learned merges:", merges)
print("Final vocab:", final_vocab)

# Apply BPE to a new word
test_word = "lower"
bpe_tokens = apply_bpe(test_word, merges)
print(f"BPE tokenization for '{test_word}':", bpe_tokens)

Learned merges: [('l', 'o'), ('lo', 'w'), ('w', 'i'), ('wi', 'l'), ('wil', 'd')]
Final vocab: {'low': 1, 'low e r': 1, 'n e w': 1, 'wild e s t': 1, 'wild': 1}
BPE tokenization for 'lower': ['low', 'e', 'r']


### 2. Word Piece
**Explanation:**
WordPiece is famously used in models like BERT. It is somewhat similar to BPE but often uses a slightly different scoring (likelihood-based) and the notion of a prefix marker (like ##token). A simplified version:

1. Start with a base vocabulary (often individual characters plus special tokens).
2. Try to add new subwords that increase the likelihood of the training data the most (or, in simpler heuristic versions, reduce perplexity or simply appear frequently).
3. Continue until you reach a vocabulary size limit.

When tokenizing:
- You look for the longest matching subword (greedily) starting from the beginning of the word.
- If a piece is found in the vocabulary, you mark subsequent subwords (if any) with ## to indicate they’re continuations of a word.

**Concrete mini-example**
- We start with an initial vocab: `["[UNK]", "l", "o", "w", "e", "r", "n", "i", "d", "s", "t"]`.
- We might discover that "low" is a frequent subword. 
- Then we add `"low"` to the vocab. Next, we might discover `"##er"` (meaning subword er that continues a word) is also frequent, etc. 
- The final vocab might look like:
    - `{"[UNK]", "l", "o", "w", "e", "r", "n", "i", "d", "s", "t", "low", "##er", "##est", "##new", ...}`

When tokenizing "lower", we scan from the start:
- We match "low" (since it’s in the vocab).
- Remaining is "er". We see "er" is in the vocab but as a continuation, so it becomes "##er".
- Hence, "lower" -> ["low", "##er"].

In [3]:
import collections


def build_wordpiece_vocab(corpus, vocab_size=15):
    """
    Very simplified WordPiece vocabulary builder:
    1. Start with each character in the vocabulary.
    2. Iteratively add bigrams or substrings that are most frequent until we reach vocab_size.
    """
    # 1) Collect all characters
    char_set = set("".join(corpus))
    # Start vocabulary with characters + special tokens
    vocab = set(["[UNK]"] + list(char_set))

    # Convert corpus into a list of words
    # (We assume corpus is already split; or we do corpus.split() if it's a single string)
    words = corpus

    while len(vocab) < vocab_size:
        # Count all possible substrings that start from an existing token
        # (Simplified: we only look at pairs "token + next char" to create something like "token##char")
        freq_counter = collections.Counter()

        for word in words:
            # Try all splits
            for i in range(1, len(word)):
                left = word[:i]  # e.g., "lo"
                right = word[i:]  # e.g., "wer"
                combined = left + "##" + right  # WordPiece style subword
                freq_counter[combined] += 1

        # Pick the most common combined subword to add to the vocab
        if not freq_counter:
            break

        best_subword, freq = freq_counter.most_common(1)[0]

        if freq < 2:
            # Stop if there's no sufficiently frequent subword
            break

        # Add this subword to vocab
        vocab.add(best_subword)

    return vocab


def wordpiece_tokenize(word, vocab):
    """
    Tokenize a single word using a simplified WordPiece approach (greedy).
    """
    tokens = []
    i = 0
    while i < len(word):
        # Try the longest possible substring from this position
        found = False
        # e.g., from i to the end
        for j in range(len(word), i, -1):
            sub = word[i:j]
            if i == 0:
                # If at start, we look for sub directly
                if sub in vocab:
                    tokens.append(sub)
                    i = j
                    found = True
                    break
                else:
                    # Also check if sub can appear as "X##rest" in vocab
                    for k in range(1, len(sub)):
                        left = sub[:k]
                        right = sub[k:]
                        combined = left + "##" + right
                        if combined in vocab:
                            tokens.append(combined)
                            i = j
                            found = True
                            break
                    if found:
                        break
            else:
                # If not at start, we look for "##sub" in vocab
                candidate = "##" + sub
                if candidate in vocab:
                    tokens.append(candidate)
                    i = j
                    found = True
                    break
                else:
                    # Also check partial splits
                    for k in range(1, len(sub)):
                        left = sub[:k]
                        right = sub[k:]
                        combined = left + "##" + right
                        if combined in vocab:
                            tokens.append(combined)
                            i = j
                            found = True
                            break
                    if found:
                        break
        if not found:
            # If we can't match anything, use [UNK]
            tokens.append("[UNK]")
            i += 1
    return tokens


# --- Example usage ---
corpus = ["low", "lower", "new", "widest", "wild"]
wp_vocab = build_wordpiece_vocab(corpus, vocab_size=15)
print("WordPiece Vocab:", wp_vocab)

test_word = "lower"
wp_tokens = wordpiece_tokenize(test_word, wp_vocab)
print(f"WordPiece tokenization for '{test_word}':", wp_tokens)

WordPiece Vocab: {'d', 'e', 't', 'l', 'i', 'w', 'r', 'n', 's', '[UNK]', 'o'}
WordPiece tokenization for 'lower': ['l', '[UNK]', '[UNK]', '[UNK]', '[UNK]']


**Note**: Real WordPiece training uses more advanced statistics (likelihood-based) rather than raw frequency, and the code above is just a toy illustration. In real usage (e.g., BERT), the tokens inside the vocabulary typically have the ## prefix to indicate continuation subwords, not a combined notation like "sub##word". But this shows the general idea.

### 3. SentencePiece
**Explanation:**
SentencePiece (developed by Google) is a more general framework that can train subword units directly from raw text without needing whitespace tokenization as a pre-step. It often uses either a Unigram language model or BPE under the hood. The key differences:

1. SentencePiece treats the text as a sequence of characters including spaces (if desired).
2. It learns a set of subword units (vocab) that maximizes the likelihood of the training data under a unigram model (or does BPE merges in the “BPETokenizer” variant).
3. It then tokenizes new text by finding the sequence of subword units (from the learned set) that has the highest probability.

For a small simplified example, you can think of it like:
- Start with a large initial vocab of every possible substring.
- Use an Expectation-Maximization procedure (for the Unigram model) to prune out subwords that are not helping model the text.
- End up with a final set of subwords that best explain the corpus.

Say the text is:
- `I love apples`
- `I love oranges`

1. SentencePiece might treat the entire string as “I▁love▁apples\nI▁love▁oranges” (where ▁ indicates a special “space” symbol).
2. It tries to find subwords that best explain this text. Maybe it ends up with a vocabulary like [I, ▁, love, ▁love, apples, ▁apples, oranges, ▁oranges, ...].
3. When tokenizing “I love oranges”, it might produce [I, ▁love, ▁oranges].

**Implementation** from scratch can get complicated because we’d have to implement the Unigram or BPE-based approach with a probability model. 

- Below is a very simplistic “frequency-based” SentencePiece-like approach that just merges frequently observed substrings. 
- This is only to demonstrate the concept, not a correct SentencePiece training.

In [4]:
import collections


def train_sentencepiece_like(corpus, vocab_size=15):
    """
    A super simplified SentencePiece-like trainer using
    a frequency-based approach (not the real Unigram LM).
    """
    # Combine corpus into a single string with special symbol for space, e.g. "_"
    combined_text = "_".join(corpus)  # underscores as "spaces"

    # Start with a set of single characters
    char_set = set(combined_text)
    sp_vocab = set(char_set)

    # Convert text into a list of tokens (initially characters)
    tokens = list(combined_text)

    while len(sp_vocab) < vocab_size:
        # Find most frequent pair in the tokens
        pairs = collections.Counter()
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pairs[pair] += 1

        if not pairs:
            break
        best_pair, freq = pairs.most_common(1)[0]
        if freq < 2:
            break

        # Merge best_pair
        new_symbol = "".join(best_pair)
        sp_vocab.add(new_symbol)

        # Rebuild tokens by merging occurrences of this pair
        merged_tokens = []
        skip = False
        for i in range(len(tokens)):
            if skip:
                skip = False
                continue
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == best_pair:
                merged_tokens.append(new_symbol)
                skip = True
            else:
                merged_tokens.append(tokens[i])
        tokens = merged_tokens

    return list(sp_vocab), tokens


def sp_like_tokenize(text, sp_vocab):
    """
    A naive tokenization using the 'SentencePiece-like' vocab.
    We greedily match the largest symbols first.
    """
    i = 0
    output = []
    while i < len(text):
        # Try longest match
        found = False
        for length in range(min(len(text) - i, max(len(s) for s in sp_vocab)), 0, -1):
            candidate = text[i : i + length]
            if candidate in sp_vocab:
                output.append(candidate)
                i += length
                found = True
                break
        if not found:  # fallback
            output.append(text[i])
            i += 1
    return output


# --- Example usage ---
if __name__ == "__main__":
    corpus = ["low", "lower", "new", "widest", "wild"]
    sp_vocab, sp_tokens = train_sentencepiece_like(corpus, vocab_size=15)
    print("Trained SentencePiece-like Vocab:", sp_vocab)
    print("Transformed tokens after merges:", sp_tokens)

    # Tokenize a new string
    test_str = "lower"
    sp_result = sp_like_tokenize(test_str, sp_vocab)
    print(f"SentencePiece-like tokenization for '{test_str}':", sp_result)

Trained SentencePiece-like Vocab: ['d', 'e', 't', 'lo', 'l', '_w', '_wi', 'i', 'low', '_', 'w', 'r', 'n', 's', 'o']
Transformed tokens after merges: ['low', '_', 'low', 'e', 'r', '_', 'n', 'e', 'w', '_wi', 'd', 'e', 's', 't', '_wi', 'l', 'd']
SentencePiece-like tokenization for 'lower': ['low', 'e', 'r']
